## imports

In [1]:
import torch
import wandb
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from model.clip import model as clip_model, processor as clip_processor
from fairface_vit import FairFaceViT

In [3]:
from dataset.dataloader import get_dataset, get_dataloaders
from dataset.transforms import get_train_transform, get_age_transform, get_val_transform

In [4]:
from training.train import train_loop, test_loop

In [5]:
from training.losses import get_age_weights, get_loss_function

## hyperparameters and WandB initialization

In [ ]:
epochs         = 20
batch_size     = 32
learning_rate  = 1e-4
weight_decay   = 1e-2
age_dropout1   = 0.15
age_hidden_dim = 384 
loss_weights  = {
    "gender":1,
    "age":1,
    "race":1
}

wandb.init(
    project="fairface-vit",
    name="clip-vit-b16-baseline",
    config={
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "age_dropout1": age_dropout1,
        "age_hidden_dim": age_hidden_dim
        "weight_decay": weight_decay,
        "optimizer": "AdamW",
        "model": "CLIP ViT-B/16",
        "loss_weights": loss_weights
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ashkanrn/.netrc.
wandb: Currently logged in as: ashkanrn (ashkanrn-university-of-guilan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Model and Device

In [ ]:
fairface_clip_model = FairFaceViT(clip_model, age_dropout1, age_hidden_dim)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
fairface_clip_model.to(device)

## Dataset and Dataloader

In [9]:
train_set, val_set = get_dataset(
    get_train_transform(clip_processor.image_processor),
    get_age_transform(clip_processor.image_processor),
    get_val_transform(clip_processor.image_processor)
)


train_dataloader, val_dataloader = get_dataloaders(
    train_set,
    val_set,
    batch_size=batch_size
)

## Loss functions


In [10]:
age_labels = np.array(train_set.dataset["age"])

age_weights = get_age_weights(
    age_labels,
    num_classes=9,
    device=device
)

loss_funcs = get_loss_function(age_weights)

## Optimizer

In [11]:

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, fairface_clip_model.parameters()),
    lr=learning_rate,
    weight_decay=weight_decay
)


## Traning

In [ ]:
best_acc = 0

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}")

    train_loss, train_task_loss = train_loop(
        train_dataloader, fairface_clip_model,
        loss_funcs, loss_weights, optimizer, device
    )

    val_loss, val_task_loss, metrics, subgroup_metrics = test_loop(
        val_dataloader,fairface_clip_model, loss_funcs, 
        loss_weights, device
    )

    log_dict = {

        "epoch": epoch + 1,
        "lr": optimizer.param_groups[0]["lr"],
        
        # losses
        "train/loss": train_loss,
        "val/loss": val_loss,


        "train/gender_loss": train_task_loss["gender"],
        "train/age_loss": train_task_loss["age"],
        "train/race_loss": train_task_loss["race"],


        "val/gender_loss": val_task_loss["gender"],
        "val/age_loss": val_task_loss["age"],
        "val/race_loss": val_task_loss["race"],
    }

    # overall metrics
    for task, values in metrics.items():
        for metric_name, value in values.items():
            log_dict[f"val/{task}/{metric_name}"] = value

    # subgroup accuracy
    for group_name, values in subgroup_metrics.items():
        for subgroup, acc in values.items():
            log_dict[f"subgroup/{group_name}/{subgroup}"] = acc

    wandb.log(log_dict)

    current_acc = (
        metrics["gender"]["accuracy"]
        +
        metrics["age"]["accuracy"]
        +
        metrics["race"]["accuracy"]
    ) / 3

    if current_acc > best_acc:
        best_acc = current_acc
        torch.save(fairface_clip_model.state_dict(), "best_model.pth")

        print(f"Saved best model | avg accuracy={best_acc:.4f}")

wandb.finish()